In [3]:
import os

os.makedirs("data", exist_ok=True)

print("Project data folder created.")

Project data folder created.


In [4]:
import csv

with open ("data/appointments.csv", newline="" ,encoding = "utf-8") as f:
    reader = csv.DictReader(f)
    print ("Headers:" , reader.fieldnames)
    rows = list(reader)
    print ("Total rows:" ,len(rows))

def parse_row(row):

    converted_row = row.copy()

    converted_row["age"] = int(converted_row["age"])
    converted_row["fee_inr"] = int(converted_row["fee_inr"])
    converted_row["prior_no_shows"] = int(converted_row["prior_no_shows"])
    converted_row["distance_km"] = float(converted_row["distance_km"])

    return converted_row

accepted = []
rejected = []

for row in rows:

  try:
    converted_row = parse_row(row)
    accepted.append(converted_row)

  except ValueError:
    rejected.append(row)

print("Accepted records:", len(accepted))
print("Rejected records:", len(rejected))

import json

with open("rejected_rows.json", "w", encoding= "utf-8") as f:
    json.dump(rejected, f, indent=2)

print("rejected record saved successfully")

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020
Accepted records: 988
Rejected records: 32
rejected record saved successfully


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/appointments.csv")

print("Dataset shape:", df.shape)

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print(df["booking_channel"].value_counts())
print("Distinct channels:", df["booking_channel"].nunique())

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")

df["booked_on"] = pd.to_datetime(df["booked_on"])

df = df.dropna(subset=["appt_date", "status"])

print("Rows after removing missing dates and statuses:", len(df))

df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

Q1 = df["distance_km"].quantile(0.25)
Q3 = df["distance_km"].quantile(0.75)

IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

upper_fence = Q3 + (1.5 * IQR)
lower_fence = Q1 - (1.5 * IQR)

print("Upper fence:", upper_fence)
print("Lower fence:", lower_fence)

distance_outliers = (
    (df["distance_km"] > upper_fence) |
    (df["distance_km"] < lower_fence)
)

print("Distance values to be capped:", distance_outliers.sum())

df["distance_km"] = np.clip(
    df["distance_km"],
    lower_fence,
    upper_fence
)

Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

upper_fence_fee = Q3_fee + (1.5 * IQR_fee)
lower_fence_fee = Q1_fee - (1.5 * IQR_fee)

print("Q1:", Q1_fee)
print("Q3:", Q3_fee)
print("IQR:", IQR_fee)

print("Upper fence:", upper_fence_fee)
print("Lower fence:", lower_fence_fee)

fee_outliers = (
    (df["fee_inr"] > upper_fence_fee) |
    (df["fee_inr"] < lower_fence_fee)
)

print("Fee values to be capped:", fee_outliers.sum())

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fence_fee,
    upper_fence_fee
)

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Rows:", len(df))

print("Missing ages:", df["age"].isnull().sum())
print("Missing distances:", df["distance_km"].isnull().sum())

print("Age data type:", df["age"].dtype)
print("Distance data type:", df["distance_km"].dtype)
print("Fee data type:", df["fee_inr"].dtype)

print("Maximum fee:", df["fee_inr"].max())


df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print(df[["status", "no_show"]].head())

Dataset shape: (1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patie

In [6]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print("Feature matrix shape:", X.shape)
print("Column means:", X.mean(axis=0).round(2))

lead_mask = df["lead_days"] > 14

print("Appointments booked more than 14 days ahead:", lead_mask.sum())
print("Share of appointments:", lead_mask.mean().round(3))
print("Percentage:", lead_mask.mean().round(3) * 100)
distance = X[:, 1]

distance_mean = np.mean(distance)
distance_std = np.std(distance)

z_scores = (distance - distance_mean) / distance_std

unusual_distances = np.abs(z_scores) > 2

print("Mean distance:", distance_mean)
print("Standard deviation:", distance_std)
print("Rows with absolute z-score above 2:", unusual_distances.sum())

Feature matrix shape: (990, 4)
Column means: [ 48.17   9.5  635.71  11.82]
Appointments booked more than 14 days ahead: 296
Share of appointments: 0.299
Percentage: 29.9
Mean distance: 9.50090909090909
Standard deviation: 6.157202334477497
Rows with absolute z-score above 2: 53


In [8]:
import pandas as pd
import requests

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 18.52,
        "longitude": 73.85,
        "hourly": "temperature_2m",
        "forecast_days": 1
    },
    timeout=10
)

print(response.status_code)

weather_data = response.json()
print(weather_data.keys())

weather_data["hourly"]["temperature_2m"]

temp_c = weather_data["hourly"]["temperature_2m"][0]

clinics = pd.read_csv("data/clinics.csv")

print(clinics.shape)
print(clinics.columns)
print(clinics.head())

def get_temperature(clinic):

    for attempt in range(2):

        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": clinic["latitude"],
                    "longitude": clinic["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            weather_data = response.json()

            hourly_data = weather_data.get("hourly", {})
            temperatures = hourly_data.get("temperature_2m")

            if not temperatures:
                reason = "Hourly temperature data missing"
                continue

            return temperatures[0]

        except requests.exceptions.RequestException as e:
            reason = str(e)

    print(f"Failed for {clinic['clinic_name']}: {reason}")
    return None

temperatures = []

for index, clinic in clinics.iterrows():

    temp = get_temperature(clinic)

    temperatures.append(temp)

    print(clinic["clinic_name"], ":", temp)

clinics["temp_c"] = temperatures

clinics.to_csv("clinics_enriched.csv", index=False)

print(clinics)
print("File saved successfully!")

200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
(8, 6)
Index(['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude'], dtype='object')
  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Kothrud Family Clinic : 19.1
Hadapsar Care Centre : 19.6
Sitabuldi Health Point : 18.9
Malviya Nagar Clinic : 14.7
Vaishali Nagar Clinic : 14.7
Edappally Medical Centre : 27.1
Vijay Nagar Clinic : 17.1
Gomti Nagar Care Point : 14.4
  clinic_id               clinic_name     city    tier  latitude  long

In [9]:
required_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

print("Unique appointment IDs:", df["appointment_id"].nunique())
print("Total rows:", len(df))

print("\nMissing values:")
print(df[required_columns].isna().sum())

print("\nTotal missing values:", df[required_columns].isna().sum().sum())

appointments_export = df[required_columns].copy()

appointments_export = appointments_export.sort_values(
    by="appointment_id"
)

print(appointments_export.shape)
print(appointments_export.head())

appointments_export["booked_on"] = (
    appointments_export["booked_on"].dt.strftime("%Y-%m-%d")
)

appointments_export["appt_date"] = (
    appointments_export["appt_date"].dt.strftime("%Y-%m-%d")
)

appointments_export.to_csv(
    "appointments_clean.csv",
    index=False
)

print("Appointments CSV exported successfully!")

Unique appointment IDs: 990
Total rows: 990

Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0
(990, 16)
  appointment_id patient_id clinic_id  booked_on  appt_date  age gender  \
0          A0001       P509       C05 2025-01-24 2025-01-25   60      F   
1          A0002       P046       C07 2025-05-27 2025-06-06   39      M   
2          A0003       P009       C02 2025-03-24 2025-03-30   63      F   
4          A0004       P482       C01 2025-04-18 2025-04-22   52      F   
5          A0005       P251       C01 2024-11-23 2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850          

In [13]:
def sql_value(value, numeric=False):

    if value is None or value == "":
        return "NULL"

    if numeric:
        return value

    return f"'{value}'"


import csv

clinic_numeric_columns = [
    "latitude",
    "longitude",
    "temp_c"
]

with open("clinics_enriched.csv", "r", encoding="utf-8") as f:
    clinic_reader = csv.DictReader(f)
    clinic_rows = list(clinic_reader)

clinic_statements = []

for clinic in clinic_rows:

    values = []

    for column in clinic:

        value = sql_value(
            clinic[column],
            numeric=column in clinic_numeric_columns
        )

        values.append(value)

    statement = f"INSERT INTO clinics VALUES ({','.join(values)});"

    clinic_statements.append(statement)

print("Clinic statements:", len(clinic_statements))
print(clinic_statements[0])

appointment_numeric_columns = [
    "age",
    "distance_km",
    "fee_inr",
    "prior_no_shows",
    "lead_days",
    "no_show"
]

with open("appointments_clean.csv", "r", encoding="utf-8") as f:
    appointment_reader = csv.DictReader(f)
    appointment_rows = list(appointment_reader)

appointment_statements = []

for appointment in appointment_rows:

    values = []

    for column in appointment:

        value = sql_value(
            appointment[column],
            numeric=column in appointment_numeric_columns
        )

        values.append(value)

    statement = f"INSERT INTO appointments VALUES ({','.join(values)});"

    appointment_statements.append(statement)

print("Appointment statements:", len(appointment_statements))
print(appointment_statements[0])

all_statements = clinic_statements + appointment_statements

with open("load_data.sql", "w", encoding="utf-8") as f:

    for statement in all_statements:
        f.write(statement + "\n")

print("Total SQL statements:", len(all_statements))
print("SQL file created successfully!")

all_statements = clinic_statements + appointment_statements

Clinic statements: 8
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,19.1);
Appointment statements: 990
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
Total SQL statements: 998
SQL file created successfully!


In [16]:
import pandas as pd

appointments = pd.read_csv("appointments_clean.csv")
clinics = pd.read_csv("clinics_enriched.csv")

clinic_data = appointments.merge(
    clinics,
    on="clinic_id",
    how="left"
)

clinic_summary = clinic_data.groupby(
    ["clinic_id", "clinic_name", "city"]
).agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
).reset_index()

clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] /
    clinic_summary["appointments"]
).round(1)

clinic_summary = clinic_summary.sort_values(
    "no_show_pct",
    ascending=False
)

print(clinic_summary)

  clinic_id               clinic_name     city  appointments  no_shows  \
7       C08    Gomti Nagar Care Point  Lucknow            90        30   
1       C02      Hadapsar Care Centre     Pune           143        39   
3       C04      Malviya Nagar Clinic   Jaipur           142        38   
0       C01     Kothrud Family Clinic     Pune           160        36   
6       C07        Vijay Nagar Clinic   Indore           101        21   
5       C06  Edappally Medical Centre    Kochi           142        23   
2       C03    Sitabuldi Health Point   Nagpur            82        13   
4       C05     Vaishali Nagar Clinic   Jaipur           130        18   

   no_show_pct  
7         33.3  
1         27.3  
3         26.8  
0         22.5  
6         20.8  
5         16.2  
2         15.9  
4         13.8  


In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())

X = df[[
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows"
]].copy()

X["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

X["booking_channel"] = df["booking_channel"]
X["visit_type"] = df["visit_type"]

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

print("Feature matrix shape:", X.shape)
print("Feature columns:", X.columns.tolist())
print(X.head())

y = df["no_show"]

appointment_ids = df["appointment_id"].copy()

Dataset shape: (990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show        

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("No-shows in test set:", y_test.sum())

baseline_predictions = np.zeros(len(y_test), dtype=int)

baseline_accuracy = (baseline_predictions == y_test).mean()

print("Baseline accuracy:", round(baseline_accuracy, 4))
print(
    "Correct predictions:",
    (baseline_predictions == y_test).sum(),
    "out of",
    len(y_test)
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Scaled training shape:", X_train_s.shape)
print("Scaled testing shape:", X_test_s.shape)
print("First 5 scaled values:\n", X_train_s[:5])

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

print("Model training completed.")
print("Number of features:", len(model.coef_[0]))

# Predict the class (0 or 1)
y_pred = model.predict(X_test_s)

# Predict the probability of a no-show
y_prob = model.predict_proba(X_test_s)[:, 1]

print("First 10 predicted classes:", y_pred[:10])
print("First 10 no-show probabilities:", y_prob[:10])

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 4))
print("\nConfusion Matrix:\n", cm)
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

import pandas as pd
import numpy as np

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coefficients["Odds_Ratio_per_SD"] = np.exp(
    coefficients["Coefficient"]
)

print(coefficients.sort_values(
    by="Coefficient",
    ascending=False
).round(4))

reminder_index = list(X.columns).index("reminder_sent")

reminder_coefficient = model.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

actual_reminder_effect = reminder_coefficient / reminder_scale

reminder_odds_ratio = np.exp(actual_reminder_effect)

print("Reminder coefficient:", reminder_coefficient)
print("Reminder scaling factor:", reminder_scale)
print("Adjusted coefficient:", actual_reminder_effect)
print("Reminder odds ratio (No to Yes):", reminder_odds_ratio)
print(
    "Percentage change in estimated odds:",
    round((reminder_odds_ratio - 1) * 100, 2), "%"
)

Training features: (792, 11)
Testing features: (198, 11)
Training target: (792,)
Testing target: (198,)
No-shows in test set: 49
Baseline accuracy: 0.7525
Correct predictions: 149 out of 198
Scaled training shape: (792, 11)
Scaled testing shape: (198, 11)
First 5 scaled values:
 [[ 0.17238448  0.39123446  0.90600345  0.38021527 -0.66743659 -1.1530012
  -0.78716701  1.32287566 -0.58318224  0.84296094 -0.84296094]
 [ 1.67335748  1.53138913 -1.80764862 -1.12891982 -0.66743659 -1.1530012
  -0.78716701  1.32287566 -0.58318224 -1.18629458  1.18629458]
 [ 0.59265692 -0.40191663  0.90600345  0.26412796 -0.66743659 -1.1530012
  -0.78716701  1.32287566 -0.58318224  0.84296094 -0.84296094]
 [-0.42800473 -0.20362885 -0.17945737 -0.43239593 -0.66743659 -1.1530012
  -0.78716701 -0.75592895  1.71472986  0.84296094 -0.84296094]
 [ 0.532618    2.67154381  0.36327304 -0.89674519 -0.66743659 -1.1530012
  -0.78716701 -0.75592895  1.71472986  0.84296094 -0.84296094]]
Model training completed.
Number of fea

In [36]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create a decision tree without restricting its depth
tree_full = DecisionTreeClassifier(random_state=42)

# Train using the original training features
tree_full.fit(X_train, y_train)

# Predict on training and testing data
train_pred_tree = tree_full.predict(X_train)
test_pred_tree = tree_full.predict(X_test)

# Calculate accuracy
train_accuracy_tree = accuracy_score(y_train, train_pred_tree)
test_accuracy_tree = accuracy_score(y_test, test_pred_tree)

print("Full Tree Depth:", tree_full.get_depth())
print("Training Accuracy:", round(train_accuracy_tree, 4))
print("Testing Accuracy:", round(test_accuracy_tree, 4))

# Create a restricted decision tree
tree_limited = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the model
tree_limited.fit(X_train, y_train)

# Predictions
train_pred_limited = tree_limited.predict(X_train)
test_pred_limited = tree_limited.predict(X_test)

# Accuracy
train_acc_limited = accuracy_score(y_train, train_pred_limited)
test_acc_limited = accuracy_score(y_test, test_pred_limited)

print("Limited Tree Depth:", tree_limited.get_depth())
print("Training Accuracy:", round(train_acc_limited, 4))
print("Testing Accuracy:", round(test_acc_limited, 4))

Full Tree Depth: 15
Training Accuracy: 1.0
Testing Accuracy: 0.6717
Limited Tree Depth: 3
Training Accuracy: 0.822
Testing Accuracy: 0.7778


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, roc_auc_score

# Create the Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict classes
rf_pred = rf_model.predict(X_test)

# Predict no-show probabilities
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Accuracy:", round(rf_accuracy, 4))
print("Random Forest Recall:", round(rf_recall, 4))
print("Random Forest ROC-AUC:", round(rf_auc, 4))

# Get feature importance scores
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

# Sort from most important to least important
importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.round(4))

# Decision Tree probabilities
tree_limited_prob = tree_limited.predict_proba(X_test)[:, 1]

# Calculate metrics for the limited tree
tree_limited_recall = recall_score(y_test, test_pred_limited)
tree_limited_auc = roc_auc_score(y_test, tree_limited_prob)

# Create comparison table
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree (Depth 3)", "Random Forest"],
    "Accuracy": [
        accuracy,
        test_acc_limited,
        rf_accuracy
    ],
    "Recall": [
        recall,
        tree_limited_recall,
        rf_recall
    ],
    "ROC-AUC": [
        roc_auc,
        tree_limited_auc,
        rf_auc
    ]
})

print(comparison.round(4))

Random Forest Accuracy: 0.7475
Random Forest Recall: 0.3061
Random Forest ROC-AUC: 0.7057
                    Feature  Importance
3                 lead_days      0.2199
0                       age      0.1968
1               distance_km      0.1947
2                   fee_inr      0.1374
4            prior_no_shows      0.1057
5             reminder_sent      0.0457
7     booking_channel_Phone      0.0269
8   booking_channel_Walk-In      0.0197
9   visit_type_Consultation      0.0194
6       booking_channel_App      0.0184
10     visit_type_Follow-up      0.0153
                     Model  Accuracy  Recall  ROC-AUC
0      Logistic Regression    0.7879  0.3469   0.7889
1  Decision Tree (Depth 3)    0.7778  0.2245   0.6997
2            Random Forest    0.7475  0.3061   0.7057


In [41]:
from sklearn.model_selection import GridSearchCV

rf_tuning_model = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid_search = GridSearchCV(
    estimator=rf_tuning_model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation ROC-AUC:", round(grid_search.best_score_, 4))

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Calculate 5-fold cross-validation ROC-AUC
lr_cv = cross_val_score(
    lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tree_cv = cross_val_score(
    tree_limited, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_cv = cross_val_score(
    rf_model, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv = grid_search.best_score_

# Compare all four models
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree (Depth 3)",
        "Random Forest (100 trees)",
        "Tuned Random Forest"
    ],
    "Cross_Validation_ROC_AUC": [
        lr_cv,
        tree_cv,
        rf_cv,
        tuned_rf_cv
    ]
})

print(cv_comparison.round(4))

Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best cross-validation ROC-AUC: 0.7673
                       Model  Cross_Validation_ROC_AUC
0        Logistic Regression                    0.7737
1    Decision Tree (Depth 3)                    0.6784
2  Random Forest (100 trees)                    0.7466
3        Tuned Random Forest                    0.7673


In [42]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Generate out-of-fold probabilities using training data
oof_prob = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare two thresholds
for threshold in [0.50, 0.35]:

    # Convert probabilities into class predictions
    oof_pred = (oof_prob >= threshold).astype(int)

    # Calculate evaluation metrics
    p = precision_score(y_train, oof_pred)
    r = recall_score(y_train, oof_pred)
    f = f1_score(y_train, oof_pred)

    # Count appointments flagged as potential no-shows
    flagged = oof_pred.sum()

    print("\nThreshold:", threshold)
    print("Precision:", round(p, 4))
    print("Recall:", round(r, 4))
    print("F1 Score:", round(f, 4))
    print("Appointments flagged:", flagged)


Threshold: 0.5
Precision: 0.5753
Recall: 0.2485
F1 Score: 0.3471
Appointments flagged: 73

Threshold: 0.35
Precision: 0.513
Recall: 0.4675
F1 Score: 0.4892
Appointments flagged: 154


In [43]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score
)

# Train the final Logistic Regression pipeline
lr_pipe.fit(X_train, y_train)

# Get no-show probabilities for the test set
final_probs = lr_pipe.predict_proba(X_test)[:, 1]

# Apply our chosen threshold
final_threshold = 0.35
final_predictions = (final_probs >= final_threshold).astype(int)

# Evaluate final model
final_accuracy = accuracy_score(y_test, final_predictions)
final_precision = precision_score(y_test, final_predictions)
final_recall = recall_score(y_test, final_predictions)
final_f1 = f1_score(y_test, final_predictions)
final_auc = roc_auc_score(y_test, final_probs)

# Count appointments flagged
final_flagged = final_predictions.sum()

print("Final Model: Logistic Regression")
print("Chosen Threshold:", final_threshold)
print("Accuracy:", round(final_accuracy, 4))
print("Precision:", round(final_precision, 4))
print("Recall:", round(final_recall, 4))
print("F1 Score:", round(final_f1, 4))
print("ROC-AUC:", round(final_auc, 4))
print("Appointments flagged:", final_flagged)
print("Total test appointments:", len(y_test))

Final Model: Logistic Regression
Chosen Threshold: 0.35
Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1 Score: 0.5714
ROC-AUC: 0.7889
Appointments flagged: 49
Total test appointments: 198


In [44]:
risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": final_probs,
    "flag": final_predictions
})

risk_scores.to_csv("test_risk_scores.csv", index=False)

print("Shape:", risk_scores.shape)
print("\nFirst 5 rows:")
print(risk_scores.head())

print("\nTotal appointments:", len(risk_scores))
print("Appointments flagged:", risk_scores["flag"].sum())
print("Missing values:", risk_scores.isnull().sum().sum())

Shape: (198, 4)

First 5 rows:
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0

Total appointments: 198
Appointments flagged: 49
Missing values: 0


In [45]:
!pip install -q google-genai pydantic

In [46]:
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

In [ ]:
API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client created successfully.")

In [51]:
MODEL_NAME = "gemini-3.8-flash"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = """
You are an assistant analysing patient follow-up notes
for a healthcare appointment management system.

Classify the reason for a missed appointment using only
the information provided in the note.

Do not invent facts or assume information that is not present.
Return only a response matching the required JSON schema.
"""

def build_prompt(note_text):

    prompt = f"""
    CONTEXT:
    CareLoop Clinics wants to understand why patients miss appointments.

    ROLE:
    Act as a careful healthcare operations data analyst.

    ACTION:
    Analyse the following patient follow-up note and identify
    the most likely reason for the missed appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    If the note is unclear or does not provide enough evidence,
    choose other rather than guessing.

    TONE:
    Be neutral, factual and concise. Do not make assumptions.

    FORMAT:
    Return:
    - root_cause: one allowed category
    - confidence: a number between 0 and 1
    - summary: no more than 25 words
    - recommended_action: a practical follow-up action

    Patient follow-up note:
    {note_text}
    """

    return prompt

In [57]:
def analyse_note(note_text):

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    data = json.loads(response.text)

    return data

In [58]:
import pandas as pd

notes_df = pd.read_csv("data/followup_notes.csv")

note_1 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

print(note_1)


Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [60]:
result = analyse_note(note_1)

print(result)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [61]:
models = client.models.list()

for model in models:
    if "flash" in model.name.lower():
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [66]:
MODEL_NAME = "gemini-flash-lite-latest"

print("Selected model:", MODEL_NAME)

result = analyse_note(note_1)

print(result)

Selected model: gemini-flash-lite-latest
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient missed the appointment because she never received a reminder message and forgot.', 'recommended_action': 'Reschedule the appointment for next week and verify reminder notification settings.'}


In [67]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def analyse_note_with_raw(note_text):

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    raw_response = response.text

    data = json.loads(raw_response)

    return data, raw_response

import time

notes_df = pd.read_csv("data/followup_notes.csv")

first_10_notes = notes_df.head(10)

print(first_10_notes)

results = []

for index, row in first_10_notes.iterrows():

    note_text = row["note_text"]

    result_row = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": None,
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # First check whether the note is blank
    if pd.isna(note_text) or str(note_text).strip() == "":

        result_row["root_cause"] = "other"
        result_row["confidence"] = 0.0
        result_row["summary"] = "No note recorded"
        result_row["recommended_action"] = "Re-call patient"
        result_row["validation_status"] = "skipped_blank"

    else:

        try:
            data, raw_response = analyse_note_with_raw(note_text)

            result_row["raw_response"] = raw_response

            # Copy whatever fields Gemini actually returned
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                if field in data:
                    result_row[field] = data[field]

            # Validate the structure and field values
            validated_data = FollowupInsight(**data)

            # Check that the cause belongs to our allowed list
            if validated_data.root_cause not in ALLOWED_CAUSES:
                raise ValueError("Root cause is not in the allowed list.")

            result_row["validation_status"] = "valid"

        except Exception as e:
            result_row["validation_status"] = "invalid"
            print(f"Error processing {row['note_id']}: {e}")

        time.sleep(13)

    results.append(result_row)

print("Processing completed.")

notes_analysed_df = pd.DataFrame(results)

notes_analysed_df.to_csv("notes_analysed.csv", index=False)

print(notes_analysed_df)
print("\nShape:", notes_analysed_df.shape)
print("\nValidation status:")
print(notes_analysed_df["validation_status"].value_counts())

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...
Error processing N001: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary

In [68]:
for i, result_row in enumerate(results):

    # Retry only API failures, not genuine validation failures
    if result_row["validation_status"] == "invalid" and result_row["raw_response"] == "":

        note_text = first_10_notes.iloc[i]["note_text"]
        note_id = result_row["note_id"]

        print(f"Retrying {note_id}...")

        success = False

        for attempt in range(3):

            try:
                data, raw_response = analyse_note_with_raw(note_text)

                result_row["raw_response"] = raw_response

                for field in [
                    "root_cause",
                    "confidence",
                    "summary",
                    "recommended_action"
                ]:
                    if field in data:
                        result_row[field] = data[field]

                validated_data = FollowupInsight(**data)

                if validated_data.root_cause not in ALLOWED_CAUSES:
                    raise ValueError("Root cause is not in the allowed list.")

                result_row["validation_status"] = "valid"
                success = True
                print(f"{note_id} successfully analysed.")
                break

            except Exception as e:

                if "503" in str(e):
                    wait_time = 30 * (attempt + 1)
                    print(f"Temporary server error. Waiting {wait_time} seconds.")
                    time.sleep(wait_time)

                else:
                    result_row["validation_status"] = "invalid"
                    print(f"Validation or response error for {note_id}: {e}")
                    break

        if not success and result_row["raw_response"] == "":
            result_row["validation_status"] = "api_error"
            print(f"{note_id} still needs processing.")

        time.sleep(13)

Retrying N001...
Temporary server error. Waiting 30 seconds.
N001 successfully analysed.
Retrying N006...
N006 successfully analysed.
Retrying N008...
N008 successfully analysed.
Retrying N009...
N009 successfully analysed.
Retrying N010...
N010 successfully analysed.


In [69]:
notes_analysed_df = pd.DataFrame(results)

print(notes_analysed_df[
    ["note_id", "root_cause", "validation_status"]
])

print("\nStatus counts:")
print(notes_analysed_df["validation_status"].value_counts())

  note_id             root_cause validation_status
0    N001  forgot_or_no_reminder             valid
1    N002  distance_or_transport             valid
2    N003            cost_or_fee             valid
3    N004                  other     skipped_blank
4    N005   illness_or_emergency             valid
5    N006       work_or_schedule             valid
6    N007  forgot_or_no_reminder             valid
7    N008  distance_or_transport             valid
8    N009                  other             valid
9    N010       work_or_schedule             valid

Status counts:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64


In [70]:
notes_analysed_df = pd.DataFrame(results)

notes_analysed_df.to_csv("notes_analysed.csv", index=False)

print("Shape:", notes_analysed_df.shape)
print("Missing values:\n", notes_analysed_df.isna().sum())
print("\nValidation statuses:")
print(notes_analysed_df["validation_status"].value_counts())

Shape: (10, 8)
Missing values:
 note_id               0
appointment_id        0
root_cause            0
confidence            0
summary               0
recommended_action    0
validation_status     0
raw_response          0
dtype: int64

Validation statuses:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64


In [71]:
# Test 1: Confidence exceeds the permitted maximum

try:
    bad_response_1 = FollowupInsight(
        root_cause="forgot_or_no_reminder",
        confidence=1.7,
        summary="Patient forgot appointment.",
        recommended_action="Send another reminder."
    )

except ValidationError as e:
    print("Test 1 - Confidence error:")
    print(e)


# Test 2: Required field is missing

try:
    bad_response_2 = FollowupInsight(
        root_cause="forgot_or_no_reminder",
        confidence=0.9,
        summary="Patient forgot appointment."
    )

except ValidationError as e:
    print("\nTest 2 - Missing field error:")
    print(e)

Test 1 - Confidence error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2 - Missing field error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or...nt forgot appointment.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [73]:
def test_prompt(note_text, prompt_version):

    if prompt_version == "V1":
        prompt = f"Classify this note: {note_text}"

    elif prompt_version == "V2":
        prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    data = json.loads(response.text)

    # Check whether the classification is allowed
    data["allowed_category"] = (
        data["root_cause"] in ALLOWED_CAUSES
    )

    return data

note_7 = first_10_notes.loc[
    first_10_notes["note_id"] == "N007",
    "note_text"
].iloc[0]

note_9 = first_10_notes.loc[
    first_10_notes["note_id"] == "N009",
    "note_text"
].iloc[0]

print("N007:", note_7)
print("N009:", note_9)

N007: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N009: Phone was switched off, could not reach the patient after three attempts.


In [74]:
import time

comparison_results = []

tests = [
    ("N007", note_7, "V1"),
    ("N007", note_7, "V2"),
    ("N009", note_9, "V1"),
    ("N009", note_9, "V2")
]

for note_id, note_text, version in tests:
    result = test_prompt(note_text, version)

    comparison_results.append({
        "note_id": note_id,
        "prompt_version": version,
        "root_cause": result["root_cause"],
        "confidence": result["confidence"],
        "allowed_category": result["allowed_category"],
        "summary": result["summary"]
    })

    print(note_id, version, result)

    if (note_id, version) != tests[-1][0:3:2]:
        time.sleep(13)

N007 V1 {'root_cause': 'forgot and no SMS reminder received', 'confidence': 0.95, 'summary': 'The patient forgot about the appointment because they did not receive an SMS reminder and now wants to reschedule.', 'recommended_action': "Reschedule the appointment and verify the patient's mobile number to ensure SMS reminders are received.", 'allowed_category': False}
N007 V2 {'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'The patient forgot the appointment because they did not receive an SMS reminder and now wants to rebook.', 'recommended_action': "Reschedule the appointment and verify the patient's mobile number for SMS reminders.", 'allowed_category': True}
N009 V1 {'root_cause': 'unreachable', 'confidence': 1.0, 'summary': 'Patient could not be reached after three attempts because their phone was switched off.', 'recommended_action': 'Try contacting the patient at a later time or send an alternate form of communication such as an email or mail.', 'allowed_categ

In [75]:
notes_analysed_df = pd.DataFrame(results)

notes_analysed_df = notes_analysed_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

notes_analysed_df.to_csv("notes_analysed.csv", index=False)

print("Shape:", notes_analysed_df.shape)
print("\nValidation status:")
print(notes_analysed_df["validation_status"].value_counts())

print("\nFinal note classifications:")
print(notes_analysed_df[["note_id", "root_cause", "validation_status"]])

Shape: (10, 8)

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Final note classifications:
  note_id             root_cause validation_status
0    N001  forgot_or_no_reminder             valid
1    N002  distance_or_transport             valid
2    N003            cost_or_fee             valid
3    N004                  other     skipped_blank
4    N005   illness_or_emergency             valid
5    N006       work_or_schedule             valid
6    N007  forgot_or_no_reminder             valid
7    N008  distance_or_transport             valid
8    N009                  other             valid
9    N010       work_or_schedule             valid


In [77]:
# Load the cleaned appointment data
appointments_df = pd.read_csv("appointments_clean.csv")

# Merge the analysed notes with appointment information
notes_merged = notes_analysed_df.merge(
    appointments_df[["appointment_id", "reminder_sent", "no_show"]],
    on="appointment_id",
    how="left"
)

# Compare root cause against reminder status
cause_reminder_table = pd.crosstab(
    notes_merged["root_cause"],
    notes_merged["reminder_sent"]
)

print("Merged data shape:", notes_merged.shape)
print("\nRoot cause vs reminder status:")
print(cause_reminder_table)

print("\nReminder status totals:")
print(notes_merged["reminder_sent"].value_counts(dropna=False))

Merged data shape: (10, 10)

Root cause vs reminder status:
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

Reminder status totals:
reminder_sent
No     5
Yes    5
Name: count, dtype: int64


In [65]:
MODEL_NAME = "gemini-flash-lite-latest"

test_response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Reply with the word OK.",
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=20
    )
)

print(test_response.text)

OK.
